In [ ]:
import re
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin
import pandas as pd
from datetime import datetime

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

In [ ]:


CLIENT_ID = '6037db494e7e459f9743e36800fe7c7c'
CLIENT_SECRET = 'e6dbda7d3d9444418ed7b314c956acc5'

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id=CLIENT_ID, client_secret=CLIENT_SECRET))

def get_track_audio_features(track_name, artist_name):
    """根據歌曲名稱和藝人名稱搜尋並獲取audio features"""
    try:
        # 搜尋track
        query = f"track:{track_name} artist:{artist_name}"
        results = sp.search(q=query, type='track', limit=1)

        if results['tracks']['items']:
            track_id = results['tracks']['items'][0]['id']
            # 獲取audio features
            features = sp.audio_features(track_id)[0]
            return features
        else:
            return None
    except Exception as e:
        print(f"Error getting features for {track_name} by {artist_name}: {e}")
        return None

def get_artist_genres(artist_name):
    """根據藝人名稱獲取genres"""
    try:
        results = sp.search(q=f"artist:{artist_name}", type='artist', limit=1)
        if results['artists']['items']:
            artist = results['artists']['items'][0]
            return artist.get('genres', [])
        return []
    except Exception as e:
        print(f"Error getting genres for {artist_name}: {e}")
        return []

# 為weekly_chart_df補充資料（注意：API有rate limit，建議分批處理）
def enrich_chart_data(df):
    enriched_data = []
    for idx, row in df.iterrows():
        print(f"Processing {idx+1}/{len(df)}: {row['title']} by {row['artist']}")

        # 獲取audio features
        features = get_track_audio_features(row['title'], row['artist'])

        # 獲取artist genres
        genres = get_artist_genres(row['artist'])

        # 合併資料
        enriched_row = row.to_dict()
        if features:
            enriched_row.update({
                'danceability': features['danceability'],
                'energy': features['energy'],
                'valence': features['valence'],
                'tempo': features['tempo'],
                'acousticness': features['acousticness'],
                'speechiness': features['speechiness'],
                'liveness': features['liveness'],
                'instrumentalness': features['instrumentalness']
            })
        else:
            # 如果找不到，用None填充
            for feat in ['danceability', 'energy', 'valence', 'tempo', 'acousticness', 'speechiness', 'liveness', 'instrumentalness']:
                enriched_row[feat] = None

        enriched_row['genres'] = genres
        enriched_data.append(enriched_row)

        # 避免API rate limit
        time.sleep(0.1)

    return pd.DataFrame(enriched_data)

# 執行補充（注意：這可能需要很長時間，取決於資料量）
enriched_df = enrich_chart_data(weekly_chart_df)
enriched_df.to_csv('spotify_enriched_data.csv', index=False)

In [ ]:
# Test Spotify API Connection
try:
    results = sp.search(q='hello world', type='track', limit=1)
    if results['tracks']['items']:
        print("成功連接Spotify API！")
        print(f"測試結果：找到歌曲 '{results['tracks']['items'][0]['name']}' 由 {results['tracks']['items'][0]['artists'][0]['name']}")
    else:
        print("連接成功，但未找到結果。")
except Exception as e:
    print(f"連接失敗：{e}")
    print("請檢查CLIENT_ID和CLIENT_SECRET是否正確設定。")

In [ ]:
import requests
import pandas as pd

artists = []

for artist_id in artist_ids:

    url = f"https://api.spotify.com/v1/artists/{artist_id}"
    
    r = requests.get(url, headers=headers)
    data = r.json()

    artists.append({
        "artist_name": data["name"],
        "artist_genres": data["genres"],
        "followers": data["followers"]["total"]
    })

df = pd.DataFrame(artists)

In [ ]:
import pandas as pd
import numpy as np
import re
import ast

# 假設 df 已經存在
# df columns example: country, artist_name, artist_genres, streams

def parse_genres(x):
    """
    Convert genre field into a Python list of lowercase strings.
    Supports:
    - list
    - string like "['pop', 'dance pop']"
    - comma-separated string
    - empty / NaN
    """
    if pd.isna(x):
        return []
    
    if isinstance(x, list):
        return [str(i).strip().lower() for i in x if str(i).strip()]
    
    if isinstance(x, str):
        x = x.strip()
        
        # string representation of list
        if x.startswith('[') and x.endswith(']'):
            try:
                parsed = ast.literal_eval(x)
                if isinstance(parsed, list):
                    return [str(i).strip().lower() for i in parsed if str(i).strip()]
            except:
                pass
        
        # comma-separated fallback
        return [i.strip().lower() for i in x.split(',') if i.strip()]
    
    return []

df = df.copy()
df["genre_list"] = df["artist_genres"].apply(parse_genres)

# optional: inspect
print(df[["country", "artist_name", "genre_list"]].head())

See what genre we have

In [ ]:
from collections import Counter

genre_counter = Counter()

for genres in df["genre_list"]:
    genre_counter.update(genres)

genre_freq = (
    pd.DataFrame(genre_counter.items(), columns=["genre", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print(genre_freq.head(100))

Mapping generes

In [ ]:
GENRE_GROUP_RULES = {
    "Pop": [
        "pop", "dance pop", "electropop", "indie pop", "art pop", "synthpop",
        "dream pop", "pop rap", "pop rock", "bedroom pop", "viral pop"
    ],
    "Hip-Hop / Rap": [
        "hip hop", "hip-hop", "rap", "trap", "drill", "grime", "boom bap",
        "gangster rap", "melodic rap", "emo rap", "phonk"
    ],
    "Rock / Alternative / Metal / Punk": [
        "rock", "alt-rock", "alternative", "alternative rock", "indie rock",
        "punk", "punk rock", "metal", "heavy metal", "black metal", "death metal",
        "hard rock", "grunge", "emo", "shoegaze", "garage rock", "post-punk"
    ],
    "R&B / Soul / Funk": [
        "r&b", "rb", "soul", "neo soul", "funk", "motown", "quiet storm", "contemporary r&b"
    ],
    "Electronic / Dance": [
        "edm", "electronic", "dance", "house", "deep house", "techno", "trance",
        "dubstep", "drum and bass", "dnb", "electro", "club", "disco", "chillstep",
        "future bass", "hardstyle"
    ],
    "Latin": [
        "latin", "reggaeton", "latin pop", "urbano latino", "latin hip hop",
        "bachata", "salsa", "cumbia", "corridos", "regional mexican", "mexican",
        "banda", "mariachi", "tropical"
    ],
    "Jazz / Blues": [
        "jazz", "blues", "swing", "bebop", "big band", "smooth jazz", "vocal jazz"
    ],
    "Classical / Instrumental / Ambient": [
        "classical", "instrumental", "ambient", "piano", "orchestral", "opera",
        "soundtrack", "film score", "neo-classical", "new age", "meditation"
    ],
    "Folk / Acoustic / Singer-Songwriter": [
        "folk", "acoustic", "singer-songwriter", "indie folk", "americana",
        "roots", "folk-pop"
    ],
    "Country / Bluegrass / Americana": [
        "country", "bluegrass", "alt-country", "americana", "honky tonk"
    ],
    "Reggae / Ska / Dub / Afro": [
        "reggae", "ska", "dub", "dancehall", "afrobeat", "afrobeats", "highlife",
        "afropop", "soca"
    ],
    "Asian Pop / Regional / World": [
        "k-pop", "kpop", "j-pop", "jpop", "mandopop", "cantopop", "c-pop", "thai pop",
        "indonesian pop", "v-pop", "opm", "anime", "city pop", "world", "world music"
    ]
}

def assign_genre_group(single_genre: str):
    """
    Map one fine-grained genre tag to a broad genre group using keyword matching.
    """
    g = single_genre.lower().strip()
    
    for big_group, keywords in GENRE_GROUP_RULES.items():
        for kw in keywords:
            if kw in g:
                return big_group
    
    return "Other"

# test
test_genres = ["k-pop girl group", "dance pop", "emo rap", "deep house", "mandopop", "vocal jazz"]
for tg in test_genres:
    print(tg, "->", assign_genre_group(tg))